# Cliff & Breakout Board: walk-forward backtest (step I1b)

Thin notebook: it only reads the outputs of `uv run twm board backtest` (PROJECT_SPEC 8.6;
definitions in `docs/board.md`): the dataset `data/board/dataset.parquet` and the tables in
`reports/board/`. Rows are players at the end-of-season snapshot of S; labels come from S+1.
Test snapshots 2007-2024 (labels 2008-2025), each fold trained on earlier snapshots only.

In [ ]:
import io
import os

import polars as pl
from IPython.display import Image, display
from matplotlib.figure import Figure

from twm.config import ROOT

os.chdir(ROOT)  # every path below is relative to the project folder
pl.Config.set_tbl_hide_dataframe_shape(True)
pl.Config.set_tbl_rows(40)
pl.Config.set_float_precision(3)
REP = ROOT / "reports/board"
dataset = pl.read_parquet(ROOT / "data/board/dataset.parquet")
metrics = pl.concat([pl.read_csv(REP / f"{p}.csv") for p in ("cliff", "breakout")])
seasons = pl.concat(
    [pl.read_csv(REP / f"{p}_seasons.csv") for p in ("cliff", "breakout")], how="diagonal"
)
features = pl.concat([pl.read_csv(REP / f"{p}_features.csv") for p in ("cliff", "breakout")])

## Populations and positives per test season

`cliff_main` trains on the Cliff rows with 6+ games in S+1; `cliff_missed` is the separate
`y_missed` label (fewer than 6 games); `cliff_sensitivity` counts a missed season as a cliff.

In [ ]:
seasons.pivot(on="variant", index="season", values="positives").sort("season")

## Label rates by age (Cliff rows, every labelled snapshot)

The aging effect the models start from: the share of top-36 veterans who fall off (or miss
most of) the next season, by age after the season.

In [ ]:
cliff = dataset.filter(pl.col("in_cliff") & pl.col("y_missed").is_not_null())
(
    cliff.with_columns(pl.col("age").floor().clip(22, 34).alias("age_band"))
    .group_by("position", "age_band")
    .agg(
        pl.len().alias("rows"),
        pl.col("y_cliff").mean().alias("cliff_rate"),
        pl.col("y_missed").mean().alias("missed_rate"),
    )
    .sort("position", "age_band")
)

## Headline: every test season (labels 2008-2025)

The models' own probabilities; 95% intervals resample whole seasons.

In [ ]:
head = metrics.filter(
    (pl.col("slice") == "all")
    & pl.col("vs").is_null()
    & pl.col("metric").is_in(["pr_auc", "p_at_10", "p_at_20"])
)
head.pivot(on="metric", index=["variant", "model"], values="value")

In [ ]:
fig = Figure(figsize=(8, 3.5))
ax = fig.subplots()
pr = head.filter(pl.col("metric") == "pr_auc").with_row_index("i")
ax.errorbar(pr["i"], pr["value"], yerr=[pr["value"] - pr["lo"], pr["hi"] - pr["value"]], fmt="o")
names = [f"{v} / {m}" for v, m in zip(pr["variant"], pr["model"], strict=True)]
ax.set_xticks(pr["i"].to_list(), names, rotation=90, fontsize=7)
ax.set_ylabel("PR-AUC (95% season-block CI)")
fig.tight_layout()
buf = io.BytesIO()
fig.savefig(buf, format="png", dpi=110)
display(Image(buf.getvalue()))

## Paired differences against the baselines

Model minus baseline on the same rows; `share_above_zero` = share of season resamples in which
the model was ahead. `ecr_era` = snapshots 2019-2024 (labels 2020-2025), where FantasyPros'
preseason ECR (taken months after the snapshot; ECR, not ADP) is the market baseline.

In [ ]:
metrics.filter(pl.col("metric").is_in(["pr_auc_diff", "p_at_10_diff"])).select(
    "variant", "slice", "model", "vs", "metric", "value", "lo", "hi", "share_above_zero"
).sort("variant", "slice", "metric", "model", "vs")

## Top features

Mean share of each model's importance over the 18 folds (logit: spread of the feature's
log-odds contribution; LightGBM: gain).

In [ ]:
features.sort("variant", "model", "mean_share", descending=[False, False, True]).group_by(
    "variant", "model", maintain_order=True
).head(6)

## Later snapshots: post-draft (I2b) and preseason (I2a, I2-fix)

The same rows and labels read later: after the S+1 draft (config `as_of.board.post_draft`, June 1
of S+1: draft capital added at his position) and at the preseason snapshot, anchor
`week1_kickoff_eve` (config default, owner decision 2026-10-02: one hour before the first
week-1 kickoff of S+1, with the week-1 depth chart) or the alternative `tuesday_before_week_1`
(spec 6.1: the Tuesday 14:00 UTC before that kickoff; no legacy week-1 chart is public yet, so
charts exist only for 2024+). `eos` = the end-of-season primary model on the same rows, compared
paired. Read from `reports/board/{post_draft,preseason,preseason_tuesday}_{cliff,breakout}.csv` (written by
`uv run twm board backtest --snapshot post_draft` / `--snapshot preseason [--anchor ...]`).

In [ ]:
later = pl.concat(
    [
        pl.read_csv(f"reports/board/{snap}_{pop}.csv").with_columns(pl.lit(snap).alias("snapshot"))
        for snap in ("post_draft", "preseason", "preseason_tuesday")
        for pop in ("cliff", "breakout")
        if os.path.exists(f"reports/board/{snap}_{pop}.csv")
    ],
    how="vertical",
)
later.filter(pl.col("metric") == "pr_auc_diff", pl.col("vs").is_in(["eos", "ecr"])).select(
    "snapshot", "variant", "slice", "model", "vs", "value", "lo", "hi"
).sort("snapshot", "variant", "slice", "model", "vs")

Top inputs of each later snapshot's primary models (the new features end in `_pd` /
`_s1`).

In [ ]:
later_features = pl.concat(
    [
        pl.read_csv(f"reports/board/{snap}_{pop}_features.csv").with_columns(
            pl.lit(snap).alias("snapshot")
        )
        for snap in ("post_draft", "preseason", "preseason_tuesday")
        for pop in ("cliff", "breakout")
        if os.path.exists(f"reports/board/{snap}_{pop}_features.csv")
    ],
    how="vertical",
)
later_features.filter(pl.col("model") == "logit").sort(
    "snapshot", "variant", "mean_share", descending=[False, False, True]
).group_by("snapshot", "variant", maintain_order=True).head(5)